# Wind VISION — Colab Pro Day 0: Trusted c-LFM migration

**Purpose:** move the *existing* NASA c-LFM research to Colab Pro without losing source/checkpoint provenance, verify PhysicsNeMo installation, and measure a first GPU hardware diagnostic. This notebook **does not** yet port c-LFM to PhysicsNeMo or benchmark a PhysicsNeMo implementation.

**Before starting:** Select **Runtime → Change runtime type → NVIDIA GPU**. Use an existing Colab Pro balance only; **do not buy compute units**. Run cells from top to bottom. External cloud transfer requires authorization for each file; the official independent test dataset is intentionally excluded from the transfer bundle.

The public NASA reference is pinned to commit `921e666c8d6060ae13bfd901dcc7f74f819d0eea`. Our local patches remain in a separate `snapshot/` area until reviewed; they are never silently overlaid on that reference.


In [12]:
import os, sys, subprocess, json, platform, time, hashlib, pathlib, shutil
from datetime import datetime, timezone
import torch
print('UTC:', datetime.now(timezone.utc).isoformat())
print('Python:', sys.version.split()[0]); print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p=torch.cuda.get_device_properties(i)
        print('GPU', i, p.name, 'VRAM GiB', round(p.total_memory/2**30,2))
    subprocess.run(['nvidia-smi'], check=False)
else:
    raise RuntimeError('No CUDA GPU. Select a GPU runtime before using paid Pro compute units.')
RUN_DIR=pathlib.Path('/content/clfm_day0_run'); RUN_DIR.mkdir(parents=True,exist_ok=True)
ENV={'utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}
(RUN_DIR/'environment_initial.json').write_text(json.dumps(ENV,indent=2))

UTC: 2026-10-29T15:46:44.524952+00:00
Python: 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
CUDA build: 13.0
CUDA available: True
GPU 0 NVIDIA L4 VRAM GiB 22.03


180

## Install minimal dependencies and PhysicsNeMo

The reference NASA package has a minimal `setup.py` without dependency pins. Colab's built-in PyTorch will be retained, and the PhysicsNeMo core install is attempted separately. If it fails, preserve the install output and **do not report a successful PhysicsNeMo setup**.

PhysicsNeMo version `2.2.2` is pinned for this initial environment. We deliberately do not install CUDA extras that could replace Colab's functioning PyTorch runtime without review.

In [13]:
import subprocess, sys, importlib.util
packages=['lightning', 'torchdyn', 'einops', 'h5py', 'scipy', 'matplotlib']
subprocess.run([sys.executable,'-m','pip','install','-q',*packages], check=True)
proc=subprocess.run([sys.executable,'-m','pip','install','-q','nvidia-physicsnemo==2.2.2'],
                    text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
physicsnemo_ok=(proc.returncode==0)
print('PhysicsNeMo pip return code:',proc.returncode)
print(proc.stdout[-9000:])
if physicsnemo_ok:
    try:
        import physicsnemo
        print('PHYSICSNEMO_IMPORT_OK:',getattr(physicsnemo,'__version__','unknown'))
    except Exception as exc:
        physicsnemo_ok=False
        print('PHYSICSNEMO_IMPORT_FAILED:',repr(exc))
(RUN_DIR/'physicsnemo_install.json').write_text(json.dumps({
    'success':physicsnemo_ok,'pip_return_code':proc.returncode,
    'tail_log':proc.stdout[-16000:], 'utc':datetime.now(timezone.utc).isoformat()
},indent=2))
if not physicsnemo_ok:
    print('IMPORTANT: Environment is NOT yet ready for the required PhysicsNeMo port.')

PhysicsNeMo pip return code: 0

PHYSICSNEMO_IMPORT_OK: 2.2.2


## Clone the unmodified NASA reference at a fixed commit

The repo does not include the large wind data files. The locally generated training file and exact research checkpoints are transferred separately. Do **not** use `wind_test_data.hdf5` for training, tuning, or model selection.

In [14]:
from pathlib import Path
REPO=Path('/content/nasa_clfm_reference')
COMMIT='921e666c8d6060ae13bfd901dcc7f74f819d0eea'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','-q','https://github.com/nasa/random_field_c-lfm.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q',COMMIT],check=True)
sha=subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],text=True,capture_output=True,check=True).stdout.strip()
assert sha == COMMIT,(sha,COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
print('Pinned NASA reference:',sha)
print('Reference encoder file:',REPO/'clfm/nn/unet1d.py')
print('Local modifications will remain separate from this checkout.')

Pinned NASA reference: 921e666c8d6060ae13bfd901dcc7f74f819d0eea
Reference encoder file: /content/nasa_clfm_reference/clfm/nn/unet1d.py
Local modifications will remain separate from this checkout.


In [15]:

from pathlib import Path
import zipfile
import os

bundle = Path("/content/drive/MyDrive/CLFM_CLOUD_BUNDLE.zip")

print("=== WIND VISION CLOUD TRANSFER CHECK ===")
print("Google Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Research bundle exists:", bundle.is_file())

if bundle.is_file():
    print("Bundle size (MB):", round(bundle.stat().st_size / 1024**2, 2))

    with zipfile.ZipFile(bundle) as archive:
        files = archive.namelist()
        print("Files in bundle:", len(files))
        print("Archive integrity:", "PASS" if archive.testzip() is None else "FAILED")

        for name in files[:30]:
            print(" ", name)

    print("\nReady to inspect and extract the research bundle.")
else:
    print("\nBundle not found.")
    print("Generate CLFM_CLOUD_BUNDLE.zip on your Mac,")
    print("then upload it to the root of Google Drive MyDrive.")

print("\nNo model training performed.")

=== WIND VISION CLOUD TRANSFER CHECK ===
Google Drive mounted: True
Research bundle exists: True
Bundle size (MB): 1003.71
Files in bundle: 15
Archive integrity: PASS
  clfm/nn/unet1d.py
  clfm/problems/wind.py
  research/scripts/vae_residual_weight_pilot.py
  research/scripts/vae_residual_pilot_field_fidelity.py
  results/control_vae_residual_weight_pilot/comparison.json
  results/control_vae_residual_pilot_field_fidelity/comparison.json
  results/control_reference_residual_interference/comparison.json
  results/research_common_grid_model_comparison/comparison.json
  results/research_next_phase_audit/vae_residual_experiment_plan.json
  results/research_vae_checkpoint_compatibility/original_reference_smoke_test.json
  results/control_reference_vae_resume_epoch10_cpu64_v2/checkpoints/epoch-epoch=0009-step=12500.ckpt
  data/wind/wind_train_data.hdf5
  transfer_manifest.json
  metadata/git_status.txt
  metadata/local_reference_diff.patch

Ready to inspect and extract the research bundle.


## Connect Drive and extract the Mac research bundle

On your Mac, run `prepare_clfm_cloud_bundle.py` (see README). Upload its `CLFM_CLOUD_BUNDLE.zip` to **MyDrive**. The bundle includes the 10-epoch checkpoint, selected provenance/experiment reports, and, if expressly approved, the training HDF5. It never includes the independent test file.

Google Drive mounting requires your direct permission. This cell verifies each included file's SHA-256 before accepting it.

In [16]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BUNDLE_PATH=Path('/content/drive/MyDrive/CLFM_CLOUD_BUNDLE.zip')
assert BUNDLE_PATH.is_file(),f'Upload the Mac bundle to {BUNDLE_PATH}, or update BUNDLE_PATH in this cell.'
import zipfile, hashlib, shutil
SNAPSHOT=Path('/content/clfm_snapshot')
SNAPSHOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    names=archive.namelist()
    for name in names:
        p=Path(name)
        if p.is_absolute() or '..' in p.parts:
            raise ValueError(f'Unsafe path in archive: {name}')
    manifest=json.loads(archive.read('transfer_manifest.json'))
    assert not manifest.get('independent_test_data_included'), 'A test dataset must not be in this bundle.'
    archive.extractall(SNAPSHOT)
for rec in manifest['files']:
    path=SNAPSHOT/rec['path']
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(chunk)
    assert h.hexdigest()==rec['sha256'], f'Checksum mismatch: {rec["path"]}'
print('Validated',len(manifest['files']),'file checksums')
print('Training file present:',(SNAPSHOT/'data/wind/wind_train_data.hdf5').is_file())
print('Reference checkpoint present:',any('epoch-epoch=0009-step=12500.ckpt' in p.name for p in SNAPSHOT.rglob('*.ckpt')))
print('Independent test excluded:',not list(SNAPSHOT.rglob('wind_test_data.hdf5')))
print('Local source versions are at',SNAPSHOT/'clfm')
print('Compare source before using as a patch:')
for rel in ('clfm/nn/unet1d.py','clfm/problems/wind.py'):
    old=(REPO/rel).read_text()
    new=(SNAPSHOT/rel).read_text() if (SNAPSHOT/rel).exists() else ''
    print(rel,'matches pinned reference:',old==new)
assert not list(SNAPSHOT.rglob('wind_test_data.hdf5'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Validated 12 file checksums
Training file present: True
Reference checkpoint present: True
Independent test excluded: True
Local source versions are at /content/clfm_snapshot/clfm
Compare source before using as a patch:
clfm/nn/unet1d.py matches pinned reference: False
clfm/problems/wind.py matches pinned reference: False


## Trusted reference checkpoint: in-memory restoration and CUDA encoder/decoder smoke test

This uses `weights_only=False` **only** for the known, locally created checkpoint, guarded by the expected SHA-256. Python pickle deserialization can execute code; do not use this mechanism with arbitrary downloaded checkpoints.

The original encoder uses a 128-feature projection; we explicitly avoid importing the modified mean+std pooling encoder. Training and physics residual parity must still be checked later.

In [17]:
import sys, torch, hashlib, warnings
CKPT=SNAPSHOT/'results/control_reference_vae_resume_epoch10_cpu64_v2/checkpoints/epoch-epoch=0009-step=12500.ckpt'
EXPECTED='b78f9e00e623ccd3b72cc3f07751de5d7f27a930e95bcabe437aada14a62eb68'
assert CKPT.is_file(),f'Checkpoint missing from approved bundle: {CKPT}'
assert hashlib.sha256(CKPT.read_bytes()).hexdigest()==EXPECTED,'Untrusted/mismatched checkpoint; refuse load'
sys.path.insert(0,str(REPO))
from clfm.nn.vae import FunctionalVAE
from clfm.nn.unet1d import Encoder1d
print('Original architecture projection input:',Encoder1d(in_channels=40,latent_dim=128).latent_projection.in_features)
model=FunctionalVAE.load_from_checkpoint(str(CKPT), map_location='cpu', weights_only=False)
assert model._encoder.latent_projection.in_features==128
model=model.to(device='cuda',dtype=torch.float32).eval()
torch.manual_seed(20261029)
with torch.inference_mode():
    synthetic=torch.zeros((2,40,256),device='cuda',dtype=torch.float32)
    mu, logvar=model.encode(synthetic)
    lo=model.grid.low.to(device='cuda',dtype=torch.float32)
    hi=model.grid.high.to(device='cuda',dtype=torch.float32)
    coords=lo+(hi-lo)*torch.rand(2,1024,4,device='cuda',dtype=torch.float32)
    values=model.decode(mu,coords)
    torch.cuda.synchronize()
assert mu.shape==logvar.shape==(2,128)
assert torch.isfinite(values).all() and torch.isfinite(mu).all()
print('CUDA ENCODER/DECODER SMOKE: PASS')
print('mu shape:',tuple(mu.shape),'decoded shape:',tuple(values.shape))
print('WARNING: Synthetic smoke test, not a reconstruction/physics or generation benchmark')

Original architecture projection input: 128


/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'encoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'branch' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['branch'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'trunk' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['trunk'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'grid' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them usin

CUDA ENCODER/DECODER SMOKE: PASS
mu shape: (2, 128) decoded shape: (2, 1024, 1)


## First matched CPU–GPU training-kernel diagnostic

This comparison is run on a **single fixed synthetic input**, same original checkpoint, **float32**, same coordinate count (`10240`), and includes **reconstruction loss + backward only**; it deliberately excludes physical/statistical residuals and optimizer updates. It is *not* the published full c-LFM training workload or a PhysicsNeMo-versus-PyTorch result. We measure medians with device synchronization and record GPU model/VRAM and warm-up. CPU results can be slower.


In [18]:
import copy, time, statistics
from pathlib import Path
torch.manual_seed(20261029)
base_model=model.to('cpu').float().eval()
with torch.inference_mode():
    low=base_model.grid.low.float().cpu(); high=base_model.grid.high.float().cpu()
U=torch.randn(1,40,256,dtype=torch.float32)*0.02+0.78
X=low+(high-low)*torch.rand(1,10240,4,dtype=torch.float32)
Y=torch.randn(1,10240,1,dtype=torch.float32)*0.02+0.78

def benchmark(device,n_warm=1,n_steps=3):
    net=copy.deepcopy(base_model).to(device=device,dtype=torch.float32).train()
    u,x,y=U.to(device),X.to(device),Y.to(device)
    times=[]
    for k in range(n_warm+n_steps):
        net.zero_grad(set_to_none=True)
        if device=='cuda': torch.cuda.synchronize()
        t0=time.perf_counter()
        mu,lv=net.encode(u)
        decoded=net.decode(mu,x).reshape_as(y)
        loss=torch.mean((decoded-y)**2)
        loss.backward()
        if device=='cuda': torch.cuda.synchronize()
        dt=time.perf_counter()-t0
        if k>=n_warm: times.append(dt)
    result={'device':device,'dtype':'float32','batch':1,'observation_shape':list(U.shape),
            'coordinate_points':10240,'loss':'synthetic reconstruction only',
            'includes_backward':True,'includes_optimizer_step':False,'includes_physics_residual':False,
            'warmups':n_warm,'repetitions':n_steps,'seconds':times,'median_seconds':statistics.median(times)}
    del net
    if device=='cuda': torch.cuda.empty_cache()
    return result

cpu_result=benchmark('cpu')
gpu_result=benchmark('cuda')
report={'experiment':'day0_reconstruction_kernel_hardware_probe','utc':datetime.now(timezone.utc).isoformat(),
        'purpose':'Hardware diagnostic only: no PhysicsNeMo comparison',
        'gpu_name':torch.cuda.get_device_name(0),
        'torch':torch.__version__,'physicsnemo_installed':physicsnemo_ok,
        'reference_commit':COMMIT,'checkpoint_sha256':EXPECTED,
        'cpu':cpu_result,'gpu':gpu_result,
        'speedup_cpu_over_gpu':cpu_result['median_seconds']/gpu_result['median_seconds'],
        'limitations':['synthetic data','float32 may differ from original float64 experiment','no residual','no optimizer step','not a full training benchmark','no physicsnemo port']}
(RUN_DIR/'reconstruction_kernel_probe.json').write_text(json.dumps(report,indent=2))
print('CPU median seconds:',round(cpu_result['median_seconds'],3))
print('GPU median seconds:',round(gpu_result['median_seconds'],3))
print('CPU/GPU kernel speed ratio (NOT PhysicsNeMo speedup):',round(report['speedup_cpu_over_gpu'],2))
print('Diagnostic output:',RUN_DIR/'reconstruction_kernel_probe.json')

CPU median seconds: 0.059
GPU median seconds: 0.028
CPU/GPU kernel speed ratio (NOT PhysicsNeMo speedup): 2.07
Diagnostic output: /content/clfm_day0_run/reconstruction_kernel_probe.json


## Persist results and finish the session

Save after every research milestone. If Drive is unavailable or full, use Colab's file download workflow. Never assume `/content` survives a disconnected runtime. Do not run the GPU unless actively measuring/training.

In [19]:
PERSIST=Path('/content/drive/MyDrive/CLFM_Research_Results')
PERSIST.mkdir(parents=True,exist_ok=True)
for item in RUN_DIR.glob('*.json'):
    shutil.copy2(item,PERSIST/item.name)
print('Copied reports to:',PERSIST)
print('Files:',[p.name for p in PERSIST.glob('*.json')])
print('NEXT SCIENTIFIC GATE: verify reference loss/gradient numerical parity and complete physics residual on CUDA BEFORE any PhysicsNeMo speedup claim.')
print('Do NOT upload or evaluate independent test data during model development.')

Copied reports to: /content/drive/MyDrive/CLFM_Research_Results
Files: ['reconstruction_kernel_probe.json', 'environment_initial.json', 'physicsnemo_install.json']
NEXT SCIENTIFIC GATE: verify reference loss/gradient numerical parity and complete physics residual on CUDA BEFORE any PhysicsNeMo speedup claim.
Do NOT upload or evaluate independent test data during model development.


In [20]:

from pathlib import Path
import json

run = Path("/content/clfm_day0_run")

print("=== WIND VISION DAY 0 RESULTS ===")

for name in [
    "environment_initial.json",
    "physicsnemo_install.json",
    "reconstruction_kernel_probe.json"
]:
    path = run / name

    print(f"\n{name}")

    if path.exists():
        data = json.loads(path.read_text())

        if name == "physicsnemo_install.json":
            print("PhysicsNeMo installed:", data.get("success"))
        elif name == "reconstruction_kernel_probe.json":
            print("GPU:", data.get("gpu_name"))
            print("CPU median:", data["cpu"]["median_seconds"], "seconds")
            print("GPU median:", data["gpu"]["median_seconds"], "seconds")
            print("CPU/GPU ratio:", data["speedup_cpu_over_gpu"])
        else:
            print(json.dumps(data, indent=2))
    else:
        print("NOT YET CREATED")

print("\nNext: full reference-loss and numerical-parity testing.")

=== WIND VISION DAY 0 RESULTS ===

environment_initial.json
{
  "utc": "2026-10-29T15:46:44.787904+00:00",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "gpu": "NVIDIA L4"
}

physicsnemo_install.json
PhysicsNeMo installed: True

reconstruction_kernel_probe.json
GPU: NVIDIA L4
CPU median: 0.058624534999580646 seconds
GPU median: 0.028378308999890578 seconds
CPU/GPU ratio: 2.0658219980551586

Next: full reference-loss and numerical-parity testing.


In [21]:

import torch
import time
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from clfm.nn.vae import reparameterize, kl_divergence

assert torch.cuda.is_available()

ckpt_path = Path(
    "/content/clfm_snapshot/results/"
    "control_reference_vae_resume_epoch10_cpu64_v2/"
    "checkpoints/epoch-epoch=0009-step=12500.ckpt"
)
expected_hash = "b78f9e00e623ccd3b72cc3f07751de5d7f27a930e95bcabe437aada14a62eb68"

with ckpt_path.open("rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == expected_hash

from clfm.nn.vae import FunctionalVAE

net = FunctionalVAE.load_from_checkpoint(
    str(ckpt_path),
    map_location="cpu",
    weights_only=False
).to(device="cuda", dtype=torch.float32)

net.train()
net.zero_grad(set_to_none=True)

torch.manual_seed(20261029)
torch.cuda.manual_seed_all(20261029)

u = torch.randn(
    1, 40, 256, device="cuda", dtype=torch.float32
) * 0.02 + 0.78

low = net.grid.low.to("cuda", dtype=torch.float32).reshape(1, 1, 4)
high = net.grid.high.to("cuda", dtype=torch.float32).reshape(1, 1, 4)

x = low + (high - low) * torch.rand(
    1, 10240, 4, device="cuda", dtype=torch.float32
)

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start = time.perf_counter()

results = {}

try:
    mu, logvar = net.encode(u)
    z = reparameterize(mu, logvar)

    rec = net.loss.reconstruction(net, z, x, u)
    residual, residual_metrics = net.loss.residual(net, z)
    kl = kl_divergence(mu, logvar)

    total = (
        rec
        + net.kld_weight * kl
        + net.res_weight * residual
    )

    assert torch.isfinite(total).all(), "Nonfinite total loss"

    total.backward()
    torch.cuda.synchronize()

    gradients = [
        p.grad
        for p in net.parameters()
        if p.grad is not None
    ]

    assert gradients, "No gradients produced"
    assert all(torch.isfinite(g).all().item() for g in gradients)

    results = {
        "status": "PASS",
        "reconstruction": float(rec.detach().cpu()),
        "kl_weighted": float((net.kld_weight * kl).detach().cpu()),
        "residual_weighted": float((net.res_weight * residual).detach().cpu()),
        "total_loss": float(total.detach().cpu()),
        "gradient_tensors": len(gradients),
        "gradient_l2_norm": float(torch.sqrt(sum(
            g.detach().float().square().sum() for g in gradients
        )).cpu())
    }

except Exception as exc:
    results = {
        "status": "FAILED",
        "error_type": type(exc).__name__,
        "error": str(exc)
    }

finally:
    torch.cuda.synchronize()
    results["elapsed_seconds"] = time.perf_counter() - start
    results["peak_gpu_memory_gib"] = (
        torch.cuda.max_memory_allocated() / 2**30
    )
    results["gpu"] = torch.cuda.get_device_name(0)
    results["precision"] = "float32"
    results["synthetic_data"] = True
    results["optimizer_step_performed"] = False
    results["cpu_gpu_numerical_parity_tested"] = False
    results["physicsnemo_port_tested"] = False
    results["utc"] = datetime.now(timezone.utc).isoformat()

    path = Path(
        "/content/drive/MyDrive/CLFM_Research_Results/"
        "full_vae_loss_cuda_probe.json"
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(results, indent=2))

print("\n=== FULL VAE LOSS CUDA PROBE ===")
print(json.dumps(results, indent=2))
print("Saved:", path)


=== FULL VAE LOSS CUDA PROBE ===
{
  "status": "PASS",
  "reconstruction": 0.011713007465004921,
  "kl_weighted": 2.9540458854171447e-05,
  "residual_weighted": 0.006371187511831522,
  "total_loss": 0.01811373606324196,
  "gradient_tensors": 164,
  "gradient_l2_norm": 0.159549281001091,
  "elapsed_seconds": 1.0598126999993838,
  "peak_gpu_memory_gib": 0.1143960952758789,
  "gpu": "NVIDIA L4",
  "precision": "float32",
  "synthetic_data": true,
  "optimizer_step_performed": false,
  "cpu_gpu_numerical_parity_tested": false,
  "physicsnemo_port_tested": false,
  "utc": "2026-10-29T16:02:36.878936+00:00"
}
Saved: /content/drive/MyDrive/CLFM_Research_Results/full_vae_loss_cuda_probe.json


In [22]:

import copy, time, json
from pathlib import Path
from datetime import datetime, timezone
import torch

from clfm.nn.vae import kl_divergence

assert torch.cuda.is_available()
torch.manual_seed(20261029)
torch.set_num_threads(4)

# Use identical synthetic inputs across all devices and precisions.
cpu_u = torch.randn(2, 40, 256, dtype=torch.float64) * 0.02 + 0.78

reference = copy.deepcopy(net).cpu().double().eval()

with torch.no_grad():
    low = reference.grid.low.detach().reshape(1, 1, 4).cpu()
    high = reference.grid.high.detach().reshape(1, 1, 4).cpu()

cpu_x = low + (high-low)*torch.rand(2, 10240, 4, dtype=torch.float64)

def evaluate(device, dtype):
    model = copy.deepcopy(reference).to(device=device, dtype=dtype).eval()
    model.zero_grad(set_to_none=True)

    u = cpu_u.to(device=device, dtype=dtype)
    x = cpu_x.to(device=device, dtype=dtype)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    mu, logvar = model.encode(u)

    # Deterministic latent instead of a stochastic reparameterization.
    z = mu

    reconstruction = model.loss.reconstruction(model, z, x, u)
    weighted_kl = model.kld_weight * kl_divergence(mu, logvar)
    loss = reconstruction + weighted_kl

    assert torch.isfinite(loss).all()
    loss.backward()

    if device == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    gradients = {
        name: parameter.grad.detach().cpu().double().clone()
        for name, parameter in model.named_parameters()
        if parameter.grad is not None
    }

    result = {
        "device": device,
        "dtype": str(dtype),
        "reconstruction": float(reconstruction.detach()),
        "weighted_kl": float(weighted_kl.detach()),
        "total": float(loss.detach()),
        "elapsed_seconds": elapsed,
        "gradient_tensors": len(gradients)
    }

    return result, gradients

print("=== CONTROLLED RECONSTRUCTION + KL COMPARISON ===", flush=True)

cpu64, g_cpu64 = evaluate("cpu", torch.float64)
cpu32, g_cpu32 = evaluate("cpu", torch.float32)
cuda32, g_cuda32 = evaluate("cuda", torch.float32)

def compare(a, b, ga, gb):
    assert ga.keys() == gb.keys()
    relative_loss = abs(a["total"]-b["total"]) / max(abs(a["total"]), 1e-12)

    squared_difference = 0.0
    squared_reference = 0.0

    for name in ga:
        diff = ga[name]-gb[name]
        squared_difference += float(torch.sum(diff*diff))
        squared_reference += float(torch.sum(ga[name]*ga[name]))

    return {
        "total_loss_relative_error": relative_loss,
        "gradient_relative_l2_error": (
            squared_difference**0.5 / max(squared_reference**0.5, 1e-12)
        )
    }

results = {
    "experiment": "reference_vae_deterministic_numerical_comparison",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "cpu_float64": cpu64,
    "cpu_float32": cpu32,
    "cuda_float32": cuda32,
    "cpu64_vs_cpu32": compare(cpu64, cpu32, g_cpu64, g_cpu32),
    "cpu32_vs_cuda32": compare(cpu32, cuda32, g_cpu32, g_cuda32),
    "uses_synthetic_inputs": True,
    "latent_sampling": "posterior_mean_deterministic",
    "physics_residual_included": False,
    "optimizer_step_performed": False,
    "physicsnemo_benchmark": False
}

out = Path("/content/drive/MyDrive/CLFM_Research_Results")
out.mkdir(parents=True, exist_ok=True)
target = out / "reference_vae_numerical_comparison.json"
target.write_text(json.dumps(results, indent=2))

print("\n=== RESULTS ===")
print("CPU float64:", cpu64["total"])
print("CPU float32:", cpu32["total"])
print("CUDA float32:", cuda32["total"])
print("\nPrecision difference:")
print(json.dumps(results["cpu64_vs_cpu32"], indent=2))
print("\nCPU/CUDA difference:")
print(json.dumps(results["cpu32_vs_cuda32"], indent=2))
print("\nSaved:", target)
print("Original model and checkpoints unchanged.")

=== CONTROLLED RECONSTRUCTION + KL COMPARISON ===

=== RESULTS ===
CPU float64: 0.011926535491449052
CPU float32: 0.011926539242267609
CUDA float32: 0.011926570907235146

Precision difference:
{
  "total_loss_relative_error": 3.14493555943201e-07,
  "gradient_relative_l2_error": 8.229958015168229e-07
}

CPU/CUDA difference:
{
  "total_loss_relative_error": 2.655000490394208e-06,
  "gradient_relative_l2_error": 1.6990009593657782e-05
}

Saved: /content/drive/MyDrive/CLFM_Research_Results/reference_vae_numerical_comparison.json
Original model and checkpoints unchanged.


In [23]:

import copy
import json
import time
import types
from pathlib import Path
from datetime import datetime, timezone

import torch

assert torch.cuda.is_available()
torch.set_num_threads(4)

# Start from the already verified reference VAE.
reference = copy.deepcopy(net).cpu().double().eval()

n_colloc = reference.loss.num_colloc
torch.manual_seed(20261029)

# One fixed latent batch; no stochastic reparameterization.
u = torch.randn(2, 40, 256, dtype=torch.float64) * 0.02 + 0.78

# Fixed normalized coordinates, generated once on CPU.
# Using the same coordinates avoids CPU/GPU RNG differences.
low = reference.grid.low.detach().cpu().double()
high = reference.grid.high.detach().cpu().double()

generator = torch.Generator(device="cpu").manual_seed(20261030)
epsilon = torch.rand(
    2 * n_colloc, 4,
    generator=generator,
    dtype=torch.float64
)

fixed_points = low + (high - low) * epsilon

def evaluate_residual(device, dtype):
    model = copy.deepcopy(reference).to(
        device=device, dtype=dtype
    ).eval()

    model.zero_grad(set_to_none=True)

    # Temporarily override sampling on this isolated model copy.
    # The pinned NASA source and original model are not modified.
    def fixed_sample(grid, count):
        assert count == len(fixed_points)
        return fixed_points.to(
            device=grid.low.device,
            dtype=grid.low.dtype
        )

    model.grid.sample = types.MethodType(
        fixed_sample, model.grid
    )

    input_u = u.to(device=device, dtype=dtype)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    mu, logvar = model.encode(input_u)

    # Fixed deterministic latent representation.
    residual, metrics = model.loss.residual(model, mu)

    assert torch.isfinite(residual).all(), (
        "Residual contains nonfinite values"
    )

    residual.backward()

    if device == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    grads = {
        name: p.grad.detach().cpu().double().clone()
        for name, p in model.named_parameters()
        if p.grad is not None
    }

    assert grads, "No residual gradients produced"
    assert all(
        torch.isfinite(g).all().item()
        for g in grads.values()
    ), "Nonfinite residual gradients"

    return {
        "device": device,
        "dtype": str(dtype),
        "residual": float(residual.detach().cpu()),
        "seconds": elapsed,
        "gradient_tensors": len(grads),
    }, grads

print("=== FIXED-COLLOCATION RESIDUAL VALIDATION ===", flush=True)

cpu64, grad64 = evaluate_residual("cpu", torch.float64)
cpu32, grad32 = evaluate_residual("cpu", torch.float32)
gpu32, grad_gpu = evaluate_residual("cuda", torch.float32)

def compare_residual(a, b, ga, gb):
    assert ga.keys() == gb.keys(), "Gradient parameters differ"

    diff_sq = 0.0
    ref_sq = 0.0

    for name in ga:
        diff_sq += float(((ga[name] - gb[name]) ** 2).sum())
        ref_sq += float((ga[name] ** 2).sum())

    return {
        "absolute_loss_error": abs(a["residual"] - b["residual"]),
        "relative_loss_error": (
            abs(a["residual"] - b["residual"])
            / max(abs(a["residual"]), 1e-12)
        ),
        "relative_gradient_l2_error": (
            diff_sq ** 0.5 / max(ref_sq ** 0.5, 1e-12)
        )
    }

report = {
    "experiment": "fixed_collocation_physics_residual_parity",
    "utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "collocation_pairs": n_colloc,
    "time_points": reference.loss.T,
    "fixed_collocation_coordinates": True,
    "deterministic_posterior_mean": True,
    "cpu_float64": cpu64,
    "cpu_float32": cpu32,
    "gpu_float32": gpu32,
    "cpu64_vs_cpu32": compare_residual(
        cpu64, cpu32, grad64, grad32
    ),
    "cpu32_vs_gpu32": compare_residual(
        cpu32, gpu32, grad32, grad_gpu
    ),
    "training_performed": False,
    "optimizer_step_performed": False,
    "physicsnemo_port_tested": False
}

out = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physics_residual_numerical_parity.json"
)
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, indent=2))

print("\n=== PHYSICS RESIDUAL RESULTS ===")
print("CPU float64:", cpu64["residual"])
print("CPU float32:", cpu32["residual"])
print("GPU float32:", gpu32["residual"])

print("\nPrecision difference:")
print(json.dumps(report["cpu64_vs_cpu32"], indent=2))

print("\nCPU/GPU difference:")
print(json.dumps(report["cpu32_vs_gpu32"], indent=2))

print("\nSaved:", out)
print("Reference checkpoint unchanged.")


=== FIXED-COLLOCATION RESIDUAL VALIDATION ===

=== PHYSICS RESIDUAL RESULTS ===
CPU float64: 0.6117757209340942
CPU float32: 0.5127544403076172
GPU float32: 0.4732172191143036

Precision difference:
{
  "absolute_loss_error": 0.09902128062647697,
  "relative_loss_error": 0.16185879438838405,
  "relative_gradient_l2_error": 453931987.48964554
}

CPU/GPU difference:
{
  "absolute_loss_error": 0.0395372211933136,
  "relative_loss_error": 0.07710751596728056,
  "relative_gradient_l2_error": 1.417169083388503
}

Saved: /content/drive/MyDrive/CLFM_Research_Results/physics_residual_numerical_parity.json
Reference checkpoint unchanged.


In [24]:

import copy, json, types, math
from pathlib import Path
from datetime import datetime, timezone
import torch

torch.set_num_threads(4)

# Reuse the inputs and reference from the immediately preceding
# fixed-collocation experiment, if still present.
assert "reference" in globals()
assert "fixed_points" in globals()
assert "u" in globals()

def residual_diagnostic(device, dtype):
    m = copy.deepcopy(reference).to(device=device, dtype=dtype).eval()
    m.zero_grad(set_to_none=True)

    points = fixed_points.detach().clone()

    def deterministic_sample(grid, count):
        assert count == points.shape[0], (count, points.shape)
        return points.to(device=grid.low.device, dtype=grid.low.dtype)

    m.grid.sample = types.MethodType(deterministic_sample, m.grid)

    source_u = u.detach().to(device=device, dtype=dtype)

    mu, _ = m.encode(source_u)
    loss, _ = m.loss.residual(m, mu)
    loss.backward()

    grads = {
        name: p.grad.detach().cpu().double().flatten()
        for name, p in m.named_parameters()
        if p.grad is not None
    }

    vector = torch.cat(list(grads.values()))

    return {
        "loss": float(loss.detach().cpu()),
        "gradient_norm": float(torch.linalg.vector_norm(vector)),
        "gradient_max_abs": float(vector.abs().max()),
        "gradient_finite": bool(torch.isfinite(vector).all()),
        "gradient_vector": vector,
        "gradient_names": list(grads)
    }

print("=== RESIDUAL REPRODUCIBILITY DIAGNOSTIC ===", flush=True)

cases = {}
for name, device, dtype in [
    ("cpu64", "cpu", torch.float64),
    ("cpu32", "cpu", torch.float32),
    ("gpu32", "cuda", torch.float32)
]:
    print("Running", name, flush=True)
    cases[name] = residual_diagnostic(device, dtype)

# Repeat the CUDA case to check whether the patched sampling
# actually produces repeatable results.
gpu_repeat = residual_diagnostic("cuda", torch.float32)

def compare(a, b):
    x, y = a["gradient_vector"], b["gradient_vector"]
    assert x.numel() == y.numel()
    return {
        "loss_absolute_difference": abs(a["loss"] - b["loss"]),
        "gradient_absolute_l2_difference": float(torch.linalg.vector_norm(x-y)),
        "reference_gradient_l2": float(torch.linalg.vector_norm(x)),
        "comparison_gradient_l2": float(torch.linalg.vector_norm(y)),
        "gradient_cosine_similarity": (
            float(torch.nn.functional.cosine_similarity(
                x.unsqueeze(0), y.unsqueeze(0)
            )) if x.norm() > 0 and y.norm() > 0 else None
        )
    }

comparisons = {
    "cpu64_vs_cpu32": compare(cases["cpu64"], cases["cpu32"]),
    "cpu32_vs_gpu32": compare(cases["cpu32"], cases["gpu32"]),
    "gpu32_repeatability": compare(cases["gpu32"], gpu_repeat)
}

summary = {
    name: {
        key: value for key, value in result.items()
        if key not in ("gradient_vector", "gradient_names")
    }
    for name, result in cases.items()
}

report = {
    "experiment": "residual_reproducibility_and_gradient_scale",
    "utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "cases": summary,
    "comparisons": comparisons,
    "fixed_collocation": True,
    "source_modified": False,
    "training_performed": False
}

path = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "residual_reproducibility_diagnostic.json"
)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(report, indent=2))

print("\n=== DIAGNOSTIC RESULTS ===")
print(json.dumps(report["cases"], indent=2))
print(json.dumps(comparisons, indent=2))
print("\nSaved:", path)


=== RESIDUAL REPRODUCIBILITY DIAGNOSTIC ===
Running cpu64
Running cpu32
Running gpu32

=== DIAGNOSTIC RESULTS ===
{
  "cpu64": {
    "loss": 0.6117757209340942,
    "gradient_norm": 2.0921273138267746e-09,
    "gradient_max_abs": 8.928022765530841e-10,
    "gradient_finite": true
  },
  "cpu32": {
    "loss": 0.5127544403076172,
    "gradient_norm": 0.9496835105190741,
    "gradient_max_abs": 0.14739209413528442,
    "gradient_finite": true
  },
  "gpu32": {
    "loss": 0.4732172191143036,
    "gradient_norm": 2.2516271890110415,
    "gradient_max_abs": 0.3778103291988373,
    "gradient_finite": true
  }
}
{
  "cpu64_vs_cpu32": {
    "loss_absolute_difference": 0.09902128062647697,
    "gradient_absolute_l2_difference": 0.9496835096467608,
    "reference_gradient_l2": 2.0921273138267746e-09,
    "comparison_gradient_l2": 0.9496835105190741,
    "gradient_cosine_similarity": 0.08723118235143522
  },
  "cpu32_vs_gpu32": {
    "loss_absolute_difference": 0.0395372211933136,
    "gradient_

In [25]:

import inspect
import json
from pathlib import Path
from datetime import datetime, timezone

assert "reference" in globals(), "Run the previous residual diagnostic first"

loss = reference.loss
grid = reference.grid

print("=== NASA PHYSICS RESIDUAL IMPLEMENTATION ===")
print("Loss class:", type(loss))
print("Grid class:", type(grid))

def show_source(label, obj, max_lines=180):
    print(f"\n=== {label} ===", flush=True)
    try:
        src = inspect.getsource(obj)
    except (OSError, TypeError) as exc:
        src = f"Source unavailable: {exc}"
    lines = src.splitlines()
    print("\n".join(
        f"{i+1:03d} {line}"
        for i, line in enumerate(lines[:max_lines])
    ))
    if len(lines) > max_lines:
        print(f"... {len(lines)-max_lines} further lines")
    return src

residual_src = show_source(
    "EXACT RESIDUAL METHOD", loss.residual
)
sample_src = show_source(
    "EXACT GRID SAMPLING METHOD", type(grid).sample
)

print("\n=== LOSS ATTRIBUTES ===")
for name in ("num_colloc", "T", "sigma_min", "eps"):
    print(name, "=", getattr(loss, name, "not defined"))

print("\n=== POSSIBLE SPECTRAL IMPLEMENTATIONS ===")
module = inspect.getmodule(type(loss))
sources = {}

if module is not None:
    for name, obj in inspect.getmembers(module):
        if inspect.isfunction(obj) and any(
            word in name.lower()
            for word in (
                "coher", "spectr", "fft",
                "residual", "correl"
            )
        ):
            sources[name] = show_source(
                f"MODULE FUNCTION: {name}", obj, 130
            )

report = {
    "utc": datetime.now(timezone.utc).isoformat(),
    "reference_commit": "921e666c8d6060ae13bfd901dcc7f74f819d0eea",
    "loss_class": str(type(loss)),
    "grid_class": str(type(grid)),
    "residual_source": residual_src,
    "grid_sample_source": sample_src,
    "additional_sources": sources
}

destination = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physics_residual_source_audit.json"
)
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2))

print("\nSaved:", destination)
print("No training performed; no source files changed.")


=== NASA PHYSICS RESIDUAL IMPLEMENTATION ===
Loss class: <class 'clfm.problems.wind.WindLoss'>
Grid class: <class 'clfm.utils.grid.RectangularGrid'>

=== EXACT RESIDUAL METHOD ===
001     def residual(self, vae, z: Tensor):
002         device = z.device
003         batch_size = z.shape[0]
004         num_colloc = self.num_colloc
005         nperseg = self.T
006 
007         # randomly select coordinates in the spatial domain. Multiply: 2* because we want pairs.
008         # strip off random dummy time component
009         x_collocation = vae.grid.sample(2 * num_colloc)[:, :3]
010 
011         # x1 = (num_colloc x 3)
012         # x2 = (num_colloc x 3)
013         # get the pairs of collocation points
014         x1, x2 = x_collocation.chunk(2, dim=0)
015 
016         # unfold into (batch x num_colloc x 1 x dim)
017         x_collocation = x_collocation.reshape(1, 2 * num_colloc, 1, 3)
018         """
019         Repeat along batch dim because signals should be measured at the same co

In [28]:

import json
from pathlib import Path
from datetime import datetime, timezone

import torch
from clfm.problems.wind import v_coherence

torch.set_num_threads(4)
generator = torch.Generator(device="cpu").manual_seed(20261029)

# Generate once so all cases use matched input signals.
A = torch.randn(16, 4, 128, generator=generator, dtype=torch.float64)
B = torch.randn(16, 4, 128, generator=generator, dtype=torch.float64)

window64 = reference.loss._hann_window.detach().cpu().double()

results = {}

print("=== NASA COHERENCE: BATCH-SIZE EXPERIMENT ===", flush=True)

for batch in [1, 2, 4, 16]:
    for device, dtype in [
        ("cpu", torch.float64),
        ("cpu", torch.float32),
        ("cuda", torch.float32),
    ]:
        x = A[:batch].to(device=device, dtype=dtype).clone()
        y = B[:batch].to(device=device, dtype=dtype).clone()
        x.requires_grad_(True)
        y.requires_grad_(True)

        window = window64.to(device=device, dtype=dtype)

        coh = v_coherence(
            x, y, nperseg=128, window=window
        )

        assert coh.shape == (4, 65), (
            f"Unexpected coherence shape: {coh.shape}"
        )

        assert torch.isfinite(coh).all(), (
            f"Nonfinite coherence: {batch}, {device}, {dtype}"
        )

        objective = coh.mean()
        objective.backward()

        assert x.grad is not None
        assert torch.isfinite(x.grad).all()

        key = f"batch{batch}_{device}_{str(dtype).split('.')[-1]}"

        results[key] = {
            "batch_size": batch,
            "device": device,
            "dtype": str(dtype),
            "coherence_mean": float(coh.detach().mean()),
            "max_deviation_from_one": float(
                (coh.detach() - 1).abs().max()
            ),
            "input_gradient_l2": float(
                torch.linalg.vector_norm(x.grad.detach().double())
            ),
            "coherence_shape": list(coh.shape)
        }

        print(
            key,
            "mean =", round(results[key]["coherence_mean"], 8),
            "max |coh-1| =", round(
                results[key]["max_deviation_from_one"], 8
            ),
            "gradient norm =",
            results[key]["input_gradient_l2"],
            flush=True
        )

report = {
    "experiment": "nasa_coherence_batch_size_and_precision",
    "utc": datetime.now(timezone.utc).isoformat(),
    "reference_commit": "921e666c8d6060ae13bfd901dcc7f74f819d0eea",
    "signal_length": 128,
    "nperseg": 128,
    "spatial_pairs": 4,
    "results": results,
    "model_checkpoint_modified": False,
    "training_performed": False
}

path = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "coherence_batch_size_precision.json"
)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(report, indent=2))

print("\nSaved:", path)
print("Original NASA implementation unchanged.")


=== NASA COHERENCE: BATCH-SIZE EXPERIMENT ===
batch1_cpu_float64 mean = 1.0 max |coh-1| = 0.0 gradient norm = 5.562485483388624e-17
batch1_cpu_float32 mean = 1.0 max |coh-1| = 2.4e-07 gradient norm = 3.323718601513109e-08
batch1_cuda_float32 mean = 1.0 max |coh-1| = 2.4e-07 gradient norm = 3.2958815879748246e-08
batch2_cpu_float64 mean = 0.47451414 max |coh-1| = 0.99932577 gradient norm = 0.041484498600353525
batch2_cpu_float32 mean = 0.47451419 max |coh-1| = 0.99932575 gradient norm = 0.04148449728101531
batch2_cuda_float32 mean = 0.47451413 max |coh-1| = 0.99932575 gradient norm = 0.041484489872179495
batch4_cpu_float64 mean = 0.24178038 max |coh-1| = 0.99962109 gradient norm = 0.02209190919953342
batch4_cpu_float32 mean = 0.2417804 max |coh-1| = 0.99962109 gradient norm = 0.022091910122413294
batch4_cuda_float32 mean = 0.24178039 max |coh-1| = 0.99962109 gradient norm = 0.022091908757752958
batch16_cpu_float64 mean = 0.0632436 max |coh-1| = 0.99981865 gradient norm = 0.0063990238328